In [29]:
import pandas as pd
import numpy as np
import random
import itertools

In [30]:
from models.classification.RandomForestClassifier import RFC
from models.classification.XGBClassifier import XGBC
from models.classification.LR import LRWrapper
from models.classification.SVC import SVCWrapper

In [31]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [32]:
from utilities import get_predictors_basic

In [33]:
from tqdm import tqdm

#### Setup grids for hyperparameter tuning

In [62]:
param_grid_rf = {
    "n_estimators": [400, 800, 1600, 3200, 6400],
    "max_depth": [None, 10, 20, 40],
    "min_samples_split": [i**2 for i in range(2, 8)],
    "min_samples_leaf": [i*2 for i in range(1, 8)],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True],
    "class_weight": ["balanced"],
    "random_state": [42]
}
param_grid_xgb = {
    "n_estimators": [400, 800, 1600, 3200, 6400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [None, 10, 20, 40],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_lambda": [1, 1.5, 2],
    "reg_alpha": [0, 0.1, 0.5],
    "random_state": [42]
}
# param_grid_lr = {
#     'penalty': ['l2', None],  # Regularization type
#     'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength (inverse of regularization)
#     'solver': ['sag', 'newton-cg', 'lbfgs'],  # Algorithm to use
#     'max_iter': [1000, 2000, 3000, 5000],  # Maximum number of iterations
#     'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
#     'class_weight': ['balanced'],  # Class weight
#     'fit_intercept': [True, False],  # Whether to include intercept
#     'warm_start': [True, False],  # Reuse the solution of the previous call to fit
#     "random_state": [42]
# }

param_grid_lr = {
    'C': [1.00000000e-04, 2.63665090e-04, 6.95192796e-04, 1.83298071e-03,
       4.83293024e-03, 1.27427499e-02, 3.35981829e-02, 8.85866790e-02,
       2.33572147e-01, 6.15848211e-01, 1.62377674e+00, 4.28133240e+00,
       1.12883789e+01, 2.97635144e+01, 7.84759970e+01, 2.06913808e+02,
       5.45559478e+02, 1.43844989e+03, 3.79269019e+03, 1.00000000e+04],
    'max_iter': [100, 1000, 2500, 5000],
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'solver': ['lbfgs', 'newton-cg', 'liblinear', 'sag', 'saga'],
    'class_weight': ['balanced', None],
}

param_grid_svc = {
    'C': [0.1, 1, 10, 100, 1000],  # Regularization parameter
    'kernel': ['linear', 'rbf'],  # Kernel type
    'degree': [2, 3, 4],  # Degree of the polynomial kernel function (only used for 'poly')
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],  # Kernel coefficient (for 'rbf', 'poly', 'sigmoid')
    'coef0': [0.0, 0.1, 0.5, 1],  # Independent term in kernel function (only used for 'poly' and 'sigmoid')
    'shrinking': [True, False],  # Whether to use the shrinking heuristic
    'class_weight': ['balanced', None],  # Class weight
    'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
    'max_iter': [1000, 2000, 3000, 5000],  # Maximum number of iterations
    "random_state": [42]
}


In [63]:
param_grids = [param_grid_rf, param_grid_xgb, param_grid_lr, param_grid_lr, param_grid_svc, param_grid_svc]

In [ ]:
model_names = [
    "RFC", "XGBC", "LogisticRegression_1vR", "LogisticRegression_1v1", "SVC_1vR", "SVC_1v1"
]

In [64]:
param_grids = [param_grid_lr, param_grid_lr, param_grid_svc, param_grid_svc]
model_names = [
    "LogisticRegression_1vR", "LogisticRegression_1v1", "SVC_1vR", "SVC_1v1"
]

#### Import data

In [36]:
matches_df = pd.read_csv('data/processed/process_ENG1.csv')

In [37]:
matches_df["xg_rolling_diff"] = abs(matches_df["xg_rolling_forHomeTeam"] - matches_df["xg_rolling_forAwayTeam"])
matches_df["xg_mean_diff"] = abs(matches_df["xg_mean_forHomeTeam"] - matches_df["xg_mean_forAwayTeam"])
matches_df["xa_rolling_diff"] = abs(matches_df["xa_rolling_forHomeTeam"] - matches_df["xa_rolling_forAwayTeam"])
matches_df["xa_mean_diff"] = abs(matches_df["xa_mean_forHomeTeam"] - matches_df["xa_mean_forAwayTeam"])
matches_df["xga_rolling_diff"] = abs(matches_df["xga_rolling_forHomeTeam"] - matches_df["xga_rolling_forAwayTeam"])
matches_df["xga_mean_diff"] = abs(matches_df["xga_mean_forHomeTeam"] - matches_df["xga_mean_forAwayTeam"])
matches_df["gf_rolling_diff"] = abs(matches_df["gf_rolling_forHomeTeam"] - matches_df["gf_rolling_forAwayTeam"])
matches_df["gf_mean_diff"] = abs(matches_df["gf_mean_forHomeTeam"] - matches_df["gf_mean_forAwayTeam"])
matches_df["ga_rolling_diff"] = abs(matches_df["ga_rolling_forHomeTeam"] - matches_df["ga_rolling_forAwayTeam"])
matches_df["ga_mean_diff"] = abs(matches_df["ga_mean_forHomeTeam"] - matches_df["ga_mean_forAwayTeam"])

In [38]:
matches_df_train, matches_df_test = train_test_split(matches_df, test_size=0.2, random_state=42, stratify=matches_df['result_code'])

In [39]:
# Print number of training and testing samples
print(f"Training samples: {len(matches_df_train)}, Testing samples: {len(matches_df_test)}")

Training samples: 672, Testing samples: 169


In [41]:
predictors = get_predictors_basic() + [
    "xg_rolling_diff",
    "xg_mean_diff",
    "xa_rolling_diff",
    "xa_mean_diff",
    "xga_rolling_diff",
    "xga_mean_diff",
    "gf_rolling_diff",
    "gf_mean_diff",
    "ga_rolling_diff",
    "ga_mean_diff"
]
# predictors = get_predictors_basic()

In [42]:
random.seed(42)
MIN_COMBINATIONS = 600

In [66]:
best_params = []
best_f1 = []
accuracies = []

for model, param_grid in tqdm(zip(model_names, param_grids), total=len(model_names), desc="Models"):
    best_model_params = None
    best_model_score = -np.inf
    accuracy = 0

    all_combinations = list(itertools.product(*param_grid.values()))

    sampled_combinations = random.sample(all_combinations, min(MIN_COMBINATIONS, len(all_combinations)))

    for combo in tqdm(sampled_combinations, desc=f"Tuning {model}"):
        params = dict(zip(param_grid.keys(), combo))
        
        try:
            if model == "RFC":
                clf = RFC(params)
            elif model == "XGBC":
                clf = XGBC(params)
            elif model == "LogisticRegression_1vR":
                clf = LRWrapper(params, one_vs_rest=True)
            elif model == "LogisticRegression_1v1":
                clf = LRWrapper(params, one_vs_rest=False)
            elif model == "SVC_1vR":
                clf = SVCWrapper(params, one_vs_rest=True)
            elif model == "SVC_1v1":
                clf = SVCWrapper(params, one_vs_rest=False)
            else:
                continue  # Unknown model, skip
        

            clf.train(matches_df_train, get_predictors_basic())

            preds = clf.evaluate_model(matches_df_test, get_predictors_basic())

            # Calculate F1 score
            f1 = f1_score(preds["Actual_Result"], preds["Predicted_Result"], average='weighted')

            # Update best parameters if current F1 is better
            if f1 > best_model_score:
                best_model_score = f1
                best_model_params = params
                accuracy = accuracy_score(preds["Actual_Result"], preds["Predicted_Result"])
        except Exception as e:
            print(f"Error initializing model {model} with params {params}: {e}")
            continue

            
    best_params.append(best_model_params)
    best_f1.append(best_model_score)
    accuracies.append(accuracy)


Models:   0%|          | 0/4 [00:00<?, ?it/s]

Error initializing model LogisticRegression_1vR with params {'C': 4.2813324, 'max_iter': 5000, 'penalty': 'elasticnet', 'solver': 'sag', 'class_weight': None}: Solver sag supports only 'l2' or 'none' penalties, got elasticnet penalty.


c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1173: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Error initializing model LogisticRegression_1vR with params {'C': 0.000695192796, 'max_iter': 1000, 'penalty': 'elasticnet', 'solver': 'lbfgs', 'class_weight': 'balanced'}: Solver lbfgs supports only 'l2' or 'none' penalties, got elasticnet penalty.
Error initializing model LogisticRegression_1vR with params {'C': 1438.44989, 'max_iter': 5000, 'penalty': 'elasticnet', 'solver': 'lbfgs', 'class_weight': None}: Solver lbfgs supports only 'l2' or 'none' penalties, got elasticnet penalty.


c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1173: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1173: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set

...

c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1173: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\negia\Desktop

Error initializing model LogisticRegression_1v1 with params {'C': 0.233572147, 'max_iter': 100, 'penalty': 'elasticnet', 'solver': 'saga', 'class_weight': None}: unsupported operand type(s) for -: 'int' and 'NoneType'


c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Error initializing model LogisticRegression_1v1 with params {'C': 0.00183298071, 'max_iter': 1000, 'penalty': 'none', 'solver': 'liblinear', 'class_weight': 'balanced'}: penalty='none' is not supported for the liblinear solver


Models:  50%|█████     | 2/4 [23:28<22:43, 681.74s/it]c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\svm\_base.py:299: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\svm\_base.py:299: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\svm\_base.py:299: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\svm\_base.py:299: Converge

In [44]:
best_f1

[0.5549618301926381,
 0.5357116282380192,
 0.5417116086064393,
 0.5668338631280958,
 0.5503089219287279,
 0.5559634029065718]

In [45]:
accuracies

[0.5798816568047337,
 0.5621301775147929,
 0.5384615384615384,
 0.5621301775147929,
 0.5680473372781065,
 0.5384615384615384]

In [46]:
best_params

[{'n_estimators': 800,
  'max_depth': None,
  'min_samples_split': 25,
  'min_samples_leaf': 12,
  'max_features': 'log2',
  'bootstrap': True,
  'class_weight': 'balanced',
  'random_state': 42},
 {'n_estimators': 3200,
  'learning_rate': 0.05,
  'max_depth': 10,
  'min_child_weight': 1,
  'subsample': 0.6,
  'colsample_bytree': 0.8,
  'gamma': 0.3,
  'reg_lambda': 1.5,
  'reg_alpha': 0.5,
  'random_state': 42},
 {'penalty': None,
  'C': 1,
  'solver': 'lbfgs',
  'max_iter': 1000,
  'tol': 0.0001,
  'class_weight': 'balanced',
  'fit_intercept': True,
  'warm_start': True,
  'random_state': 42},
 {'penalty': 'l2',
  'C': 10,
  'solver': 'sag',
  'max_iter': 2000,
  'tol': 0.01,
  'class_weight': 'balanced',
  'fit_intercept': False,
  'warm_start': True,
  'random_state': 42},
 {'C': 0.1,
  'kernel': 'sigmoid',
  'degree': 4,
  'gamma': 0.01,
  'coef0': 0.5,
  'shrinking': False,
  'class_weight': 'balanced',
  'tol': 0.001,
  'max_iter': 5000,
  'random_state': 42},
 {'C': 0.01,
  'k